# Tool-Discovery Agent

Discover bioinformatics tools from papers, inject them into your agent, and run tasks.

**Flow:** Paper discovery (PubMed) -> Extract GitHub repos -> Standardize to registry -> Inject into agent -> Run tasks

In [ ]:
# Step 0: Setup
import os, sys, subprocess
if not os.path.exists('/content/st2/agent_connector'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/caixiaoyao2025/st2.git', '/content/st2'], check=True)
ST2_DIR = '/content/st2'
sys.path.insert(0, ST2_DIR)
os.chdir(ST2_DIR)
!pip install -q langchain langchain-openai langgraph pydantic graph-tool-call 2>/dev/null || true
from IPython.display import display, Markdown
display(Markdown('**Setup done**'))

## Step 1 - Input your agent & API key

In [ ]:
import os, subprocess
from IPython.display import display, Markdown
import ipywidgets as widgets

path_input = widgets.Text(
    value='https://github.com/snap-stanford/Biomni.git',
    placeholder='Git URL or /content/my_agent',
    description='Agent:', layout=widgets.Layout(width='90%'))
key_input = widgets.Password(
    value='', placeholder='ark-... or sk-...',
    description='API Key:', layout=widgets.Layout(width='90%'))
model_input = widgets.Text(
    value='deepseek-v4-flash-ga-260731',
    description='Model:', layout=widgets.Layout(width='70%'))
base_input = widgets.Text(
    value='https://ark.cn-beijing.volces.com/api/v3',
    description='Base URL:', layout=widgets.Layout(width='90%'))
btn = widgets.Button(description='Connect agent', button_style='primary')
out = widgets.Output()

def on_connect(_):
    out.clear_output()
    with out:
        agent_src = path_input.value.strip()
        api_key = key_input.value.strip()
        model = model_input.value.strip()
        base_url = base_input.value.strip()
        if not agent_src:
            display(Markdown('**ERROR:** Enter agent path or URL')); return
        if not api_key:
            display(Markdown('**ERROR:** Enter API key')); return
        if agent_src.startswith('http'):
            agent_dir = '/content/_agent_' + agent_src.split('/')[-1].replace('.git', '')
            if not os.path.isdir(agent_dir):
                subprocess.run(['git', 'clone', '--depth', '1', agent_src, agent_dir],
                               capture_output=True, text=True)
            display(Markdown(f'Cloned to `{agent_dir}`'))
        else:
            agent_dir = agent_src
            if not os.path.isdir(agent_dir):
                display(Markdown(f'**ERROR:** `{agent_dir}` not found')); return
        os.environ['OPENAI_API_KEY'] = api_key
        os.environ['OPENAI_BASE_URL'] = base_url
        os.environ['OPENAI_MODEL'] = model
        os.environ['WESTLAKE_API_KEY'] = api_key
        get_ipython().user_ns['agent_dir'] = agent_dir
        get_ipython().user_ns['api_key_val'] = api_key
        get_ipython().user_ns['model_val'] = model
        get_ipython().user_ns['base_url_val'] = base_url
        display(Markdown(f'**Agent:** `{agent_dir}` | **Model:** `{model}` | **Key:** `{api_key[:8]}...`'))

btn.on_click(on_connect)
display(widgets.VBox([path_input, key_input, model_input, base_input, btn, out]))

## Step 2 - Scan agent & detect wiring

In [ ]:
import os, sys, json
from IPython.display import display, Markdown
import yaml

from agent_connector.scanner import build_schema

schema = build_schema(agent_dir, include_evidence=False)
lines = [
    '**Detected:**',
    f'- agent_class = `{schema.get("agent_class", "N/A")}`',
    f'- registration_method = `{schema.get("registration_method", "N/A")}`',
    f'- execution_method = `{schema.get("execution_method", "N/A")}`',
    f'- execution_candidates = `{schema.get("execution_candidates", [])}`',
    f'- wiring_style = `{schema.get("wiring_style", "N/A")}`',
]
display(Markdown('<br>'.join(lines)))

# Load registry
reg_path = os.path.join(ST2_DIR, 'data', 'mcp_registry.yaml')
tools = yaml.safe_load(open(reg_path, encoding='utf-8'))['tools']
display(Markdown(f'**Registry:** {len(tools)} tools'))

## Step 3 - Choose execution entrypoint (if ambiguous)

In [ ]:
from IPython.display import display, Markdown
import ipywidgets as widgets

candidates = schema.get('execution_candidates', [])
exec_method = schema.get('execution_method')

if len(candidates) > 1:
    display(Markdown(f'**Found {len(candidates)} execution methods.** Pick the one your agent uses:'))
    radio = widgets.RadioButtons(options=candidates, value=exec_method)
    confirm = widgets.Button(description='Confirm', button_style='primary')
    pick_out = widgets.Output()
    def on_confirm(_):
        pick_out.clear_output()
        with pick_out:
            chosen = radio.value
            get_ipython().user_ns['exec_method_override'] = chosen
            display(Markdown(f'**Using:** `{chosen}`'))
    confirm.on_click(on_confirm)
    display(widgets.VBox([radio, confirm, pick_out]))
elif exec_method:
    get_ipython().user_ns['exec_method_override'] = exec_method
    display(Markdown(f'**Execution method:** `{exec_method}` (auto-detected)'))
else:
    get_ipython().user_ns['exec_method_override'] = 'run'
    display(Markdown('**No execution method detected.** Defaulting to `run`.'))

## Step 4 - Preflight check

In [ ]:
import os
from IPython.display import display, Markdown
from agent_connector.agent_preflight import preflight, preflight_report

pf = preflight(agent_dir)
display(Markdown(preflight_report(pf)))

In [ ]:
import subprocess, os, sys
from IPython.display import display, Markdown

if pf.status == 'SETUP_REQUIRED':
    pkgs = pf.pip_installable
    if pkgs:
        display(Markdown(f'Installing **{len(pkgs)}** packages...'))
        cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + pkgs
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        display(Markdown(f'**Done** (returncode={r.returncode})'))
    else:
        display(Markdown('No auto-installable packages'))
else:
    display(Markdown(f'Status = `{pf.status}`'))

## Step 5 - Generate wiring & inject via Adapter

In [ ]:
import os, sys, json, importlib
from IPython.display import display, Markdown

from agent_connector.generator import generate_wiring, load_wrappers, load_adapter

# Apply user's execution method choice
if exec_method_override != schema.get('execution_method'):
    schema['execution_method'] = exec_method_override

wiring_dir = os.path.join(ST2_DIR, 'wiring')
wiring = generate_wiring(tools, schema, out_dir=wiring_dir)

display(Markdown(f'**Wiring mode:** `{wiring["mode"]}`'))
for k, v in wiring['artifacts'].items():
    if isinstance(v, str) and os.path.isfile(v):
        display(Markdown(f'- `{k}`: `{v}`'))
    elif isinstance(v, list):
        display(Markdown(f'- `{k}`: {len(v)} files'))

# Load wrappers
sys.path.insert(0, wiring_dir)
sys.path.insert(0, agent_dir)
reg_style = schema.get('registration_style') or 'object'
wrappers = load_wrappers(package_name='generated_tools', registration_style=reg_style)
for w in wrappers:
    if not hasattr(w, 'name'):
        w.name = getattr(w, '__name__', None)
display(Markdown(f'**{len(wrappers)} wrappers loaded**'))

# Load adapter (unified interface)
adapter_cls = load_adapter(schema.get('agent_class') or 'Agent',
                           adapter_path=wiring['artifacts']['adapter'])

# For Biomni: skip create_agent (special init), inject directly
agent = None
used_adapter = None
if schema.get('agent_class') == 'A1' or 'biomni' in agent_dir.lower():
    try:
        from biomni.agent.A1 import A1
        import io, contextlib
        f = io.StringIO()
        with contextlib.redirect_stdout(f):
            real_agent = A1(path='/content/_biomni_data', expected_data_lake_files=[])
        used_adapter = adapter_cls(agent=real_agent)
        used_adapter.register_tools(real_agent, wrappers)
        agent = real_agent
        display(Markdown(f'**Injected into Biomni A1:** {len(wrappers)} tools'))
    except Exception as e:
        display(Markdown(f'Biomni failed: `{e}`'))

# Generic agent: use adapter.create_agent()
if agent is None and schema.get('registration_method'):
    try:
        used_adapter = adapter_cls()
        agent = used_adapter.create_agent()
        used_adapter.register_tools(agent, wrappers)
        display(Markdown(f'**Injected into {type(agent).__name__}:** {len(wrappers)} tools'))
    except Exception as e:
        display(Markdown(f'Inject failed: `{e}`'))

# Fallback
if agent is None:
    class DynamicAgent:
        def __init__(self): self.tools = []
    DynamicAgent.add_tool = lambda self, t: self.tools.append(t)
    agent = DynamicAgent()
    for w in wrappers: agent.add_tool(w)
    used_adapter = adapter_cls(agent=agent)
    display(Markdown(f'**Fallback:** DynamicAgent with {len(agent.tools)} tools'))

get_ipython().user_ns['agent'] = agent
get_ipython().user_ns['wrappers'] = wrappers
get_ipython().user_ns['adapter'] = used_adapter

display(Markdown(f'**Final agent:** `{type(agent).__name__}`'))

## Step 6 - Run tasks via Agent Loop

Ask the agent a bioinformatics question. The LLM will:
1. **Retrieve** relevant tools from the registry
2. **Select** the best tool
3. **Extract** arguments from your query
4. **Execute** the tool
5. **Validate** the result
6. **Replan** if needed

In [ ]:
import os, sys, json
from IPython.display import display, Markdown

from openai import OpenAI
from agent_connector.tool_runner import run_tool_spec, format_result
from agent_connector.tool_spec import make_leaf_spec
from agent_connector.graph_retrieval import build_graph_from_tools, retrieve_tools

client = OpenAI(api_key=os.environ['OPENAI_API_KEY'], base_url=os.environ['OPENAI_BASE_URL'])
MODEL = os.environ['OPENAI_MODEL']

# Build fnmap
fnmap = {}
for t in tools:
    if t.get('arg_style') == 'subcommand' and t.get('subcommand_details'):
        for sub_name in t['subcommands']:
            leaf = make_leaf_spec(t, sub_name)
            fnmap[leaf['name']] = leaf
    else:
        fnmap[t['name']] = t

# Build function schemas for LLM
def to_fn_schema(spec):
    props = {}
    required = []
    for pname, meta in (spec.get('inputs') or {}).items():
        props[pname] = {'type': 'string', 'description': (meta or {}).get('description', '')}
        if (meta or {}).get('required'):
            required.append(pname)
    return {'type': 'function', 'function': {
        'name': spec['name'], 'description': (spec.get('description') or '')[:300],
        'parameters': {'type': 'object', 'properties': props, 'required': required}}}

schemas = [to_fn_schema(s) for s in fnmap.values()]
graph = build_graph_from_tools(tools)
display(Markdown(f'**Agent loop ready:** {len(fnmap)} tools, model `{MODEL}`'))

In [ ]:
import json, os
from IPython.display import display, Markdown
import ipywidgets as widgets
from openai import OpenAI
from agent_connector.tool_runner import run_tool_spec, format_result
from agent_connector.agent import _select_tool, extract_arguments, validate_arguments, validate_result, MAX_RETRIES
from agent_connector.graph_retrieval import retrieve_tools

client = OpenAI(api_key=os.environ['OPENAI_API_KEY'], base_url=os.environ['OPENAI_BASE_URL'])
MODEL = os.environ['OPENAI_MODEL']

query_input = widgets.Text(
    value='Analyze the FASTA file at /content/sample.fasta',
    placeholder='Ask a bioinformatics question...',
    description='Query:', layout=widgets.Layout(width='95%'))
run_btn = widgets.Button(description='Run agent', button_style='success')
result_out = widgets.Output()

def run_agent_loop(_):
    result_out.clear_output()
    with result_out:
        query = query_input.value.strip()
        if not query:
            display(Markdown('**Enter a query**')); return
        display(Markdown(f'**Query:** {query}'))
        candidates_all = [s['function']['name'] for s in schemas]
        tool_descs = {s['function']['name']: s['function'].get('description', '') for s in schemas}
        excluded = set()
        final_answer = None

        for attempt in range(MAX_RETRIES + 1):
            if graph is not None:
                results = retrieve_tools(graph, query, top_k=5)
                candidate_names = [r[0] for r in results if r[0] not in excluded]
            else:
                candidate_names = [n for n in candidates_all if n not in excluded]
            if not candidate_names:
                candidate_names = [n for n in candidates_all if n not in excluded]
            if not candidate_names:
                display(Markdown('**No candidates**')); break

            tool_name = _select_tool(query, candidate_names, client, MODEL, tool_descs)
            display(Markdown(f'**Turn {attempt+1}:** `{tool_name}`'))
            if tool_name is None:
                display(Markdown('NO_MATCHING_TOOL')); break

            spec = fnmap.get(tool_name)
            if spec is None:
                excluded.add(tool_name); continue

            args = extract_arguments(query, spec, client, MODEL)
            display(Markdown(f'**Args:** `{json.dumps(args)[:200]}`'))

            is_valid, missing = validate_arguments(spec, args)
            if not is_valid:
                display(Markdown(f'**Missing:** {missing}'))
                final_answer = f'Need: {missing}'
                break

            try:
                raw = run_tool_spec(spec, args, timeout=300)
                result_text = format_result(raw)
                exec_ok = raw.get('return_code') == 0 and raw.get('status') == 'ok'
                display(Markdown(f'**Result:** `{result_text[:300]}`'))
            except Exception as e:
                display(Markdown(f'**Error:** `{e}`'))
                excluded.add(tool_name)
                continue

            if not exec_ok:
                excluded.add(tool_name)
                continue

            satisfied, reason = validate_result(query, tool_name, result_text, client, MODEL)
            if satisfied:
                final_answer = result_text
                display(Markdown(f'**VALIDATED** - {reason}'))
                break
            else:
                display(Markdown(f'**Not satisfied:** {reason} - replanning...'))
                excluded.add(tool_name)

        display(Markdown(f'---\n### Final answer\n{final_answer or "(none)"}'))

run_btn.on_click(run_agent_loop)
display(widgets.VBox([query_input, run_btn, result_out]))